<div align="center">

# 🛍️ Shopsy Home & Kitchen Analysis

**Web Scraping • Data Cleaning • SQL Analysis**

</div>

---

**🌐 Source:** [Shopsy](https://www.shopsy.in/)

**🎯 Goal:** Scrape and analyze Home & Kitchen product data to discover useful pricing, discount, rating, and product insights.

**🛠️ Tools:** Python • Requests • BeautifulSoup • Pandas • MySQL • SQL

**🔄 Workflow:**  
**Scrape → Clean → Store → Analyze → Insights**

<div align="center">

##  Web Scraping

</div>

**Source:** Shopsy – Home & Kitchen

**Method:** Requests + BeautifulSoup

**Process:**  
**Listing Page → Product Links → Product Details → Dataset**

**Target:** Collect product information such as **Name, Price, Discount, Rating, Reviews, Brand, Capacity, Material, and URL**.

<div align="center"> 
 
###  Install Required Libraries
 
**Installing the Python libraries required for web scraping and data handling.**
 
</div>

In [15]:
#%pip install requests scrapy pandas matplotlib mysql-connector-python selenium
#%pip install beautifulsoup4

<div align="center">

###  Import Libraries

**Loading the Python libraries required for web scraping and data handling.**

</div>

In [33]:
import time
import re
import json
import requests
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from scrapy import Selector
from IPython.display import display
from mysql.connector import connect, Error
from bs4 import BeautifulSoup

<div align="center"> 
 
### Shopsy Kitchen Containers URL
 
**Defining the Shopsy Home & Kitchen product listing URL for web scraping.**
 
</div>

In [ ]:


SHOPSY_URL = "https://www.shopsy.in/household/containers-bottles/containers-jars/kitchen-containers/pr?sid=r4l%2Cv2a%2C29e%2C4cf"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/151.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9"
}


<div align="center"> 
 
### Get Listing Page
 
**Fetching the Shopsy listing page and preparing its HTML content for product extraction.**
 
</div>

In [40]:

response = requests.get(
    SHOPSY_URL,
    headers=headers,
    timeout=30
)

response.raise_for_status()

print("Listing Status:", response.status_code)

soup = BeautifulSoup(response.text, "html.parser")

Listing Status: 200


<div align="center"> 
 
###  Collect Product Links
 
**Extracting unique product URLs from the Shopsy listing page for detailed scraping.**
 
</div>

In [41]:


product_links = []
for anchor in soup.find_all("a", href=True):
    href = anchor["href"]
    if "/p/" in href:
        if href.startswith("/"):
            href = "https://www.shopsy.in" + href
        if href not in product_links:
            product_links.append(href)

product_links = list(dict.fromkeys(product_links))[:30]
print("Product links found:", len(product_links))

Product links found: 30


<div align="center"> 
 
###  Data Extraction Helper Functions
 
**Defining reusable functions to clean text and extract product details such as price, discount, rating, reviews, capacity, and material.**
 
</div>

In [34]:
def clean(text):
    if not text:
        return ""
    return re.sub(r"\s+", " ", str(text)).strip()


def find_price(text):
    for pattern in [r"₹\s*([\d,]+)", r"Rs\.?\s*([\d,]+)"]:
        match = re.search(pattern, text, re.I)
        if match:
            return "₹" + match.group(1)
    return ""


def find_discount(text):
    for pattern in [r"(\d{1,3})\s*%\s*off", r"(\d{1,3})\s*%\s*discount"]:
        match = re.search(pattern, text, re.I)
        if match:
            return match.group(1) + "%"
    return ""


def find_rating(text):
    for pattern in [r"([1-5]\.\d)\s*(?:★|Stars?|Ratings?)", r"\b([1-5]\.\d)\b"]:
        match = re.search(pattern, text, re.I)
        if match:
            value = float(match.group(1))
            if 1 <= value <= 5:
                return match.group(1)
    return ""


def find_reviews(text):
    for pattern in [r"([\d,]+)\s*Reviews?", r"([\d,]+)\s*Ratings?"]:
        match = re.search(pattern, text, re.I)
        if match:
            return match.group(1)
    return ""


def find_capacity(text):
    patterns = [
        r"\b\d+(?:\.\d+)?\s*(?:ml|ML)\b",
        r"\b\d+(?:\.\d+)?\s*(?:l|L)\b",
        r"\b\d+(?:\.\d+)?\s*(?:litre|litres)\b"
    ]
    values = []
    for pattern in patterns:
        values.extend(re.findall(pattern, text, re.I))
    return ", ".join(dict.fromkeys(values))


def find_material(text):
    materials = [
        "Stainless Steel", "Plastic", "Glass", "Steel", "Acrylic",
        "Ceramic", "Silicone", "Aluminium", "Aluminum", "Wood"
    ]
    found = [
        material for material in materials
        if re.search(r"\b" + re.escape(material) + r"\b", text, re.I)
    ]
    return ", ".join(dict.fromkeys(found))

<div align="center"> 
 
###  Scrape Each Product Page
 
**Visiting each product URL and extracting the required product information.**
 
</div>

In [ ]:


data = []

for index, product_url in enumerate(product_links, start=1):
    print(f"Scraping {index}/{len(product_links)}")

    try:
        product_response = requests.get(
            product_url,
            headers=headers,
            timeout=30
        )
        product_response.raise_for_status()
        product_soup = BeautifulSoup(product_response.text, "html.parser")
        page_text = clean(product_soup.get_text(" ", strip=True))

        jsonld_data = []
        for script in product_soup.find_all("script", type="application/ld+json"):
            try:
                value = json.loads(script.string or script.get_text())
                jsonld_data.extend(value if isinstance(value, list) else [value])
            except (TypeError, json.JSONDecodeError):
                continue

        product_name = ""
        price = ""
        rating = ""
        reviews = ""
        brand = ""

        for item in jsonld_data:
            if not isinstance(item, dict) or item.get("@type") != "Product":
                continue

            product_name = clean(item.get("name", ""))
            brand_data = item.get("brand")
            if isinstance(brand_data, dict):
                brand = clean(brand_data.get("name", ""))
            elif isinstance(brand_data, str):
                brand = clean(brand_data)

            rating_data = item.get("aggregateRating", {})
            if isinstance(rating_data, dict):
                rating = clean(rating_data.get("ratingValue", ""))
                reviews = clean(rating_data.get("reviewCount", rating_data.get("ratingCount", "")))

            offers = item.get("offers", {})
            if isinstance(offers, dict) and offers.get("price"):
                price = "₹" + str(offers["price"])
            break

        if not product_name and product_soup.title:
            product_name = re.sub(
                r"\s*[-|]\s*Shopsy.*$", "",
                clean(product_soup.title.get_text()),
                flags=re.I
            )

        if not price:
            price = find_price(page_text)
        discount = find_discount(page_text)
        rating = rating or find_rating(page_text)
        reviews = reviews or find_reviews(page_text)

        if not brand:
            brand_match = re.search(
                r"Brand\s*[:\-]?\s*([A-Za-z0-9 &._-]+)",
                page_text,
                re.I
            )
            if brand_match:
                brand = clean(brand_match.group(1))

        capacity = find_capacity(page_text) or find_capacity(product_name)
        material = find_material(page_text) or find_material(product_name)

        data.append({
            "Product_Name": product_name,
            "Price": price,
            "Discount": discount,
            "Rating": rating,
            "Reviews": reviews,
            "Brand": brand,
            "Capacity": capacity,
            "Material": material,
            "Product_URL": product_url
        })
        time.sleep(1)

    except requests.RequestException as error:
        print("Request error:", str(error)[:100])
    except Exception as error:
        print("Error:", str(error)[:100])

<div align="center"> 
 
### Create DataFrame & Save CSV
 
**Converting the scraped product data into a structured DataFrame and saving it as a CSV file.**
 
</div>

In [ ]:


df = pd.DataFrame(data).drop_duplicates(subset=["Product_URL"])
output_file = "shopsy_kitchen_products.csv"
df.to_csv(output_file, index=False, encoding="utf-8-sig")

print("\n" + "=" * 60)
print("SCRAPING COMPLETED")
print("=" * 60)
print("Products scraped:", len(df))
print("CSV file:", output_file)
display(df.head(20))


SCRAPING COMPLETED
Products scraped: 29
CSV file: shopsy_kitchen_products.csv


,Product_Name,Price,Discount,Rating,Reviews,Brand,Capacity,Material,Product_URL
0,Qtrix Pack of 8 Plastic Grocery Container - 50...,₹300,69%,4.4,104,Shopsy,"500 ml, 1100 ml, 1500 ml",Plastic,https://www.shopsy.in/qtrix-pack-8-plastic-gro...
1,AneriDEALS Pack of 24 Plastic Grocery Containe...,₹423,78%,4.1,34,Shopsy,"250 ml, 350 ml, 650 ml, 1200 ml, 1000 ml",Plastic,https://www.shopsy.in/anerideals-pack-24-plast...
2,BELIZZI Pack of 6 Plastic Fridge Container - 1...,₹257,74%,4.1,272,Shopsy,"1500 ml, 500 ml, 1000 ml",Plastic,https://www.shopsy.in/belizzi-pack-6-plastic-f...
3,SHAN Pack of 2 Ceramic Pickle Jar - 500 ml Green,₹345,50%,4.3,,Shopsy,500 ml,Ceramic,https://www.shopsy.in/shan-pack-2-ceramic-pick...
4,"RK Pack of 5 Steel Cookie Jar - 300 ml, 500 ml...",₹383,76%,4.1,83,Shopsy,"300 ml, 500 ml, 750 ml, 1150 ml, 1650 ml, 375 ...",Steel,https://www.shopsy.in/rk-pack-5-steel-cookie-j...
5,VR Pack of 3 Plastic Grocery Container - 4500 ...,₹247,75%,4.1,210,Shopsy,"4500 ml, 250 ml",Plastic,https://www.shopsy.in/vr-pack-3-plastic-grocer...
6,Loknath Pack of 12 Plastic Utility Container -...,₹567,43%,4.1,3,Shopsy,"2000 ml, 1350 ml, 4000 ml","Plastic, Steel",https://www.shopsy.in/loknath-pack-12-plastic-...
7,Hoatzin Pack of 6 Plastic Grocery Container - ...,₹497,50%,4.3,7,Shopsy,"12000 ml, 8000 ml, 6000 ml, 3000 ml, 2000 ml, ...",Plastic,https://www.shopsy.in/hoatzin-pack-6-plastic-g...
8,Veksin Pack of 6 Plastic Grocery Container - 1...,₹497,64%,4.3,23,Shopsy,"1000 ml, 2000 ml, 3000 ml, 6000 ml, 8000 ml, 1...",Plastic,https://www.shopsy.in/veksin-pack-6-plastic-gr...
9,TASTIC Pack of 1 Plastic Grocery Container - 1...,₹125,74%,3.5,0,Shopsy,1800 ml,Plastic,https://www.shopsy.in/tastic-pack-1-plastic-gr...
